This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [1]:
import great_expectations as gx
context = gx.get_context()
import logging

In [2]:
logging.basicConfig(level=logging.INFO, force = True)

In [3]:
## THIS IS REQUIRED FOR THE TECHNICAL VIEW HACK
# TODO handling of credentials not ideal, required for technical view fix
import os

from google.cloud import bigquery
import pandas as pd

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '/mnt/encrypted_data/git/api_keys/world-fishing-827-02584bdf5326.json'
client = bigquery.Client()

In [4]:
gx_temp_schema = "tech_great_expectations_temp_ttl_7d"
date_filter = "hour <= '2099-12-31'"

In [5]:
def partition_enforcement_enabled(client, schema_name: str, table_name: str):
    partition_enforcement_enabled_sql = f"""
  SELECT
    option_value
  FROM
    {schema_name}.INFORMATION_SCHEMA.TABLE_OPTIONS
  WHERE
    table_name = '{table_name}'
  AND 
    option_name = 'require_partition_filter';
    """
    
    # TODO CHO20230622 handle exceptions
    partition_enforcement_enabled_res = pd.read_gbq(partition_enforcement_enabled_sql, project_id='world-fishing-827', dialect='standard')

    # TODO CHO20230622 handle multiple rows returned
    return(partition_enforcement_enabled_res.shape[0] > 0)

In [6]:
def get_partition_cols(client, schema_name: str, table_name: str):
    get_partition_cols_sql = f"""
  SELECT
    column_name,
    data_type
  FROM
    {schema_name}.INFORMATION_SCHEMA.COLUMNS
  WHERE
    table_name = '{table_name}'
  AND
    is_partitioning_column = 'YES';
    """
    
    # TODO CHO20230622 handle exceptions
    get_partition_cols_res = pd.read_gbq(get_partition_cols_sql, project_id='world-fishing-827', dialect='standard')

    # TODO CHO20230622 handle multiple rows returned
    return(get_partition_cols_res)

In [7]:
def create_or_replace_tech_view(client, schema_name: str, table_name: str, gx_temp_schema: str, default_max_date: str = "2099-12-31", date_filter: str = None):
    # TODO CHO20230623 automatically get partition col
    if date_filter is None:
        if partition_enforcement_enabled(client, schema_name, table_name):
            partition_cols = get_partition_cols(client, schema_name, table_name)
            if partition_cols.query("data_type.isin(['TIMESTAMP', 'DATE'])").shape[0]:
                partition_col_used = partition_cols.query("data_type.isin(['TIMESTAMP', 'DATE'])")["column_name"][0]
                date_filter = f"{partition_col_used} < '{default_max_date}'"
                logging.info(f"""
                Partition enforcement enabled and found date or timestamp type column among partition columns.
                Using date filter: {date_filter}
                """)
            else:
                logging.warn(f"""
                Partition enforcement enabled but no date columns found among partitions columns {partition_date_cols['column_name']}.
                Proceeding but eventually queries will fail!
                """)
        else:
            logging.info(f"Partition enforcement not enabled, not applying date filter")
    else:
        logging.info(f"Using provided date filter: {date_filter}")

    date_filter_sql = "" if date_filter is None else f" WHERE {date_filter}"
    # TODO CHO20230622 validate parameters, e.g. date filter
    # TODO CHO20230622 check first whether view exists before replacing
    fq_view_name = f"{gx_temp_schema}.v_unfiltered_{schema_name}_{table_name}"
    query = f"""
    CREATE OR REPLACE VIEW `{fq_view_name}` AS (
    SELECT * FROM {schema_name}.{table_name} {date_filter_sql}
    )
    """

    # TODO CHO20230622 handle exceptions
    client.query(query)
    return(fq_view_name)    

In [8]:
connection_string = """bigquery://world-fishing-827/tech_great_expectations_temp_ttl_7d?\
credentials_path=/mnt/encrypted_data/git/api_keys/world-fishing-827-02584bdf5326.json"""

In [9]:
schema_name = "pipe_ais_v3_alpha_published"
            
table_name = [
    "satellite_timing_offsets", #partition filter enforcement
    "vessel_info", #no partition filter enforcement
    "segs_activity_daily" #first real world test (hours < 48)
][2]
table_name

'segs_activity_daily'

In [10]:
use_query_asset = True

In [11]:
#we create a data source for each schema, e.g. pipe_ais_v3_alpha_published
#get datasource if it exists, otherwise create datasource
# WARNING: it's necessary to distinguish because running add_or_update_sql resets the datasource config
# TODO: create feature request to simply get datasource if it already exists
if schema_name in [ds.get("name") for ds in context.list_datasources()]:
    datasource = context.get_datasource(schema_name)
else:
    datasource = context.sources.add_or_update_sql(
        name=schema_name, connection_string=connection_string, create_temp_table=True
    )

In [12]:
gx_view_name = create_or_replace_tech_view(client, schema_name, table_name, gx_temp_schema)

INFO:root:
                Partition enforcement enabled and found date or timestamp type column among partition columns.
                Using date filter: date < '2099-12-31'
                


In [13]:
gx_view_name

'tech_great_expectations_temp_ttl_7d.v_unfiltered_pipe_ais_v3_alpha_published_segs_activity_daily'

In [14]:
if table_name in datasource.get_asset_names():
    table_asset = datasource.get_asset(table_name)
elif use_query_asset:
    table_asset = datasource.add_query_asset(
        name=table_name,
        query=f"SELECT * FROM {gx_view_name}"
    )
else:
    table_asset = datasource.add_table_asset(
        name=table_name, 
        table_name=f"{schema_name}.{table_name}"
    )

In [15]:
table_asset

QueryAsset(name='segs_activity_daily', type='query', id=None, order_by=[Sorter(key='year', reverse=True), Sorter(key='month', reverse=True), Sorter(key='day', reverse=True)], batch_metadata={}, splitter=SplitterYearAndMonthAndDay(column_name='date', method_name='split_on_year_and_month_and_day'), query='SELECT * FROM tech_great_expectations_temp_ttl_7d.v_unfiltered_pipe_ais_v3_alpha_published_segs_activity_daily')

In [16]:
table_asset.add_splitter_year_and_month_and_day("date")
table_asset.add_sorters(["-year", "-month", "-day"])

INFO:great_expectations.data_context.data_context.file_data_context:Saving 3 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml


QueryAsset(name='segs_activity_daily', type='query', id=None, order_by=[Sorter(key='year', reverse=True), Sorter(key='month', reverse=True), Sorter(key='day', reverse=True)], batch_metadata={}, splitter=SplitterYearAndMonthAndDay(column_name='date', method_name='split_on_year_and_month_and_day'), query='SELECT * FROM tech_great_expectations_temp_ttl_7d.v_unfiltered_pipe_ais_v3_alpha_published_segs_activity_daily')

In [18]:
# this saves the table_asset with all its characteristics (incl splitter)
context.update_datasource(datasource=datasource)

INFO:great_expectations.data_context.data_context.file_data_context:Saving 3 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml


SQLDatasource(type='sql', name='pipe_ais_v3_alpha_published', id=None, assets=[QueryAsset(name='satellite_timing_offsets', type='query', id=None, order_by=[Sorter(key='year', reverse=True), Sorter(key='month', reverse=True), Sorter(key='day', reverse=True)], batch_metadata={}, splitter=SplitterYearAndMonthAndDay(column_name='hour', method_name='split_on_year_and_month_and_day'), query='SELECT * FROM tech_great_expectations_temp_ttl_7d.v_unfiltered_pipe_ais_v3_alpha_published_satellite_timing_offsets'), QueryAsset(name='vessel_info', type='query', id=None, order_by=[Sorter(key='year', reverse=True), Sorter(key='month', reverse=True), Sorter(key='day', reverse=True)], batch_metadata={}, splitter=SplitterYearAndMonthAndDay(column_name='first_timestamp', method_name='split_on_year_and_month_and_day'), query='SELECT * FROM tech_great_expectations_temp_ttl_7d.v_unfiltered_pipe_ais_v3_alpha_published_vessel_info'), QueryAsset(name='segs_activity_daily', type='query', id=None, order_by=[Sorter

In [19]:
br = table_asset.build_batch_request({"year": 2023, "month": 1, "day": 1})
batches = table_asset.get_batch_list_from_batch_request(br)

INFO:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.dict() - substituting config values
INFO:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.dict() - substituting config values
INFO:great_expectations.datasource.data_connector.batch_filter:batch_slice: None was parsed to: slice(0, None, None)


In [20]:
batches

[Batch(datasource=SQLDatasource(type='sql', name='pipe_ais_v3_alpha_published', id=None, assets=[QueryAsset(name='satellite_timing_offsets', type='query', id=None, order_by=[Sorter(key='year', reverse=True), Sorter(key='month', reverse=True), Sorter(key='day', reverse=True)], batch_metadata={}, splitter=SplitterYearAndMonthAndDay(column_name='hour', method_name='split_on_year_and_month_and_day'), query='SELECT * FROM tech_great_expectations_temp_ttl_7d.v_unfiltered_pipe_ais_v3_alpha_published_satellite_timing_offsets'), QueryAsset(name='vessel_info', type='query', id=None, order_by=[Sorter(key='year', reverse=True), Sorter(key='month', reverse=True), Sorter(key='day', reverse=True)], batch_metadata={}, splitter=SplitterYearAndMonthAndDay(column_name='first_timestamp', method_name='split_on_year_and_month_and_day'), query='SELECT * FROM tech_great_expectations_temp_ttl_7d.v_unfiltered_pipe_ais_v3_alpha_published_vessel_info'), QueryAsset(name='segs_activity_daily', type='query', id=None

In [21]:
for batch in batches:
    print(batch.columns())

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

  sqlalchemy.util.warn(

  sqlalchemy.util.warn(



['seg_id', 'ssvid', 'hours', 'active_hours', 'fishing_hours', 'night_loitering_hours', 'positions', 'active_positions', 'port_hours', 'avg_distance_from_shore_m', 'avg_distance_from_shore_fishing_m', 'avg_depth_m', 'avg_depth_fishing_m', 'max_lat', 'min_lat', 'max_lon', 'min_lon', 'avg_lat_lon', 'first_timestamp', 'last_timestamp', 'pos_w_speed_over_50', 'avg_speed_knots', 'terrestrial_positions', 'satellite_positions', 'terr_less_than_10mi', 'terr_more_than_150mi', 'terr_more_than_300mi', 'pos_over_3200km_from_sat', 'type', 'type.value', 'type.count', 'receivers_outofrange', 'receivers_outofrange.receiver', 'receivers_outofrange.pings', 'sat_positions_known', 'avg_sat_lat_lon', 'avg_dist_to_sat_km', 'dist_avg_pos_sat_vessel_km', 'date']


In [22]:
expectation_suite_name = f"{schema_name}.{table_name}"

In [23]:
if expectation_suite_name not in context.list_expectation_suite_names():
    context.add_or_update_expectation_suite(expectation_suite_name)

In [24]:
validator = context.get_validator(batch_request=br, expectation_suite_name=expectation_suite_name)

INFO:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.dict() - substituting config values
INFO:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.dict() - substituting config values
INFO:great_expectations.datasource.data_connector.batch_filter:batch_slice: None was parsed to: slice(0, None, None)


In [25]:
#TODO: the workflow looks different later
if table_name == "satellite_timing_offsets":
    validator.expect_column_values_to_not_be_null("receiver")
    validator.expect_column_values_to_not_be_null("avg_distance_from_sat_km")
elif table_name == "vessel_info":
    validator.expect_column_values_to_not_be_null("vessel_id")
else:
    validator.expect_column_values_to_not_be_null("hours")
    validator.expect_column_max_to_be_between("hours", min_value=0, max_value=48)

  warnings.warn(



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

In [26]:
validator.save_expectation_suite(discard_failed_expectations=False)

INFO:great_expectations.validator.validator:	2 expectation(s) included in expectation_suite.


In [27]:
checkpoint = gx.checkpoint.SimpleCheckpoint(
    name=table_name + "_hello_world_cp",
    data_context=context,
    validations=[
        {
            "batch_request": br,
            "expectation_suite_name": expectation_suite_name,
        },
    ],
)

In [28]:
checkpoint_result = checkpoint.run()

INFO:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.dict() - substituting config values
INFO:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.dict() - substituting config values
INFO:great_expectations.datasource.data_connector.batch_filter:batch_slice: None was parsed to: slice(0, None, None)
INFO:great_expectations.validator.validator:	2 expectation(s) included in expectation_suite.


Calculating Metrics:   0%|          | 0/12 [00:00<?, ?it/s]

In [29]:
checkpoint_result

{
  "run_id": {
    "run_name": null,
    "run_time": "2023-06-26T13:28:41.692430-05:00"
  },
  "run_results": {
    "ValidationResultIdentifier::pipe_ais_v3_alpha_published/segs_activity_daily/__none__/20230626T182841.692430Z/pipe_ais_v3_alpha_published-segs_activity_daily-year_2023-month_1-day_1": {
      "validation_result": {
        "evaluation_parameters": {},
        "meta": {
          "great_expectations_version": "0.17.1",
          "expectation_suite_name": "pipe_ais_v3_alpha_published.segs_activity_daily",
          "run_id": {
            "run_name": null,
            "run_time": "2023-06-26T13:28:41.692430-05:00"
          },
          "batch_spec": {
            "data_asset_name": "segs_activity_daily",
            "query": "SELECT * FROM tech_great_expectations_temp_ttl_7d.v_unfiltered_pipe_ais_v3_alpha_published_segs_activity_daily",
            "temp_table_schema_name": null,
            "batch_identifiers": {
              "date": {
                "year": 2023,
    

In [30]:
context.add_checkpoint(checkpoint=checkpoint)

  warnings.warn(



{
  "action_list": [
    {
      "name": "store_validation_result",
      "action": {
        "class_name": "StoreValidationResultAction"
      }
    },
    {
      "name": "store_evaluation_params",
      "action": {
        "class_name": "StoreEvaluationParametersAction"
      }
    },
    {
      "name": "update_data_docs",
      "action": {
        "class_name": "UpdateDataDocsAction"
      }
    }
  ],
  "batch_request": {},
  "class_name": "SimpleCheckpoint",
  "config_version": 1.0,
  "evaluation_parameters": {},
  "module_name": "great_expectations.checkpoint",
  "name": "segs_activity_daily_hello_world_cp",
  "profilers": [],
  "runtime_configuration": {},
  "validations": [
    {
      "batch_request": {
        "datasource_name": "pipe_ais_v3_alpha_published",
        "data_asset_name": "segs_activity_daily",
        "options": {
          "year": 2023,
          "month": 1,
          "day": 1
        }
      },
      "expectation_suite_name": "pipe_ais_v3_alpha_published.se

In [31]:
context.run_checkpoint(table_name + "_hello_world_cp")

INFO:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.dict() - substituting config values
INFO:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.dict() - substituting config values
INFO:great_expectations.datasource.data_connector.batch_filter:batch_slice: None was parsed to: slice(0, None, None)
INFO:great_expectations.validator.validator:	2 expectation(s) included in expectation_suite.


Calculating Metrics:   0%|          | 0/12 [00:00<?, ?it/s]

{
  "run_id": {
    "run_name": null,
    "run_time": "2023-06-26T13:29:02.851136-05:00"
  },
  "run_results": {
    "ValidationResultIdentifier::pipe_ais_v3_alpha_published/segs_activity_daily/__none__/20230626T182902.851136Z/pipe_ais_v3_alpha_published-segs_activity_daily-year_2023-month_1-day_1": {
      "validation_result": {
        "evaluation_parameters": {},
        "meta": {
          "great_expectations_version": "0.17.1",
          "expectation_suite_name": "pipe_ais_v3_alpha_published.segs_activity_daily",
          "run_id": {
            "run_name": null,
            "run_time": "2023-06-26T13:29:02.851136-05:00"
          },
          "batch_spec": {
            "data_asset_name": "segs_activity_daily",
            "query": "SELECT * FROM tech_great_expectations_temp_ttl_7d.v_unfiltered_pipe_ais_v3_alpha_published_segs_activity_daily",
            "temp_table_schema_name": null,
            "batch_identifiers": {
              "date": {
                "year": 2023,
    

In [32]:
#optional - I believe only needed when building docs for the first time
context.build_data_docs()

{'local_site': 'file:///mnt/encrypted_data/git/data-testing/great_expectations/uncommitted/data_docs/local_site/index.html'}